In [2]:
import pandas as pd

df = pd.read_csv("../../data/aggregated_rolling_features.csv")
df.head()
df.shape

(5539, 97)

In [3]:
stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]

for stat in stats:
    team1_col = f"previous_10_games_team1_average_{stat}"
    team2_col = f"previous_10_games_team2_average_{stat}"
    diff_col = f"{stat}_diff"
    
    df[diff_col] = df[team1_col] - df[team2_col]

df["map_score_diff"] = (
    df["team1_previous_10_average_map_score"] 
    - df["team2_previous_10_average_map_score"]
)

diff_features = [
    "kills_diff",
    "deaths_diff",
    "assists_diff",
    "adr_diff",
    "kast_diff",
    "kddiff_diff",
    "map_score_diff"
]

df[diff_features + ["team1_win"]].head()

,kills_diff,deaths_diff,assists_diff,adr_diff,kast_diff,kddiff_diff,map_score_diff,team1_win
0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0
1,0.6,-0.6,-3.2,0.52,1.68,1.2,0.0,1
2,1.5,-1.6,-0.8,3.11,8.47,3.1,0.0,1
3,0.0,0.0,0.0,0.00,0.00,0.0,0.0,1
4,9.0,-9.0,3.0,46.80,48.00,18.0,0.0,1


In [4]:
df.shape

(5539, 104)

In [5]:
X = df[diff_features]
y = df["team1_win"]

print(X.isna().sum())
print(y.isna().sum())

kills_diff        0
deaths_diff       0
assists_diff      0
adr_diff          0
kast_diff         0
kddiff_diff       0
map_score_diff    0
dtype: int64
0


In [6]:
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]

for stat in stats:
    df[f"{stat}_diff"] = (
        df[f"previous_10_games_team1_average_{stat}"]
        - df[f"previous_10_games_team2_average_{stat}"]
    )

df["map_score_diff"] = (
    df["team1_previous_10_average_map_score"]
    - df["team2_previous_10_average_map_score"]
)

features = [
    "kills_diff",
    "deaths_diff",
    "assists_diff",
    "adr_diff",
    "kast_diff",
    "kddiff_diff",
    "map_score_diff"
]

data = df[features + ["team1_win", "datetime"]].dropna()

split_index = int(len(data) * 2 / 3)

train = data.iloc[:split_index]
test = data.iloc[split_index:]

X_train = train[features]
y_train = train["team1_win"]

X_test = test[features]
y_test = test["team1_win"]

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

## I use ChatGTP do this pipeline
lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

grid_search = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=param_grid,
    cv=10,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

Best parameters: {'model__C': 1}
Best CV accuracy: 0.5817893503259357


In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score

best_lasso = grid_search.best_estimator_

y_pred = best_lasso.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Test Accuracy: 0.5858148348673524
Kappa: 0.12617927810091012
Confusion Matrix:
[[261 567]
 [198 821]]
              precision    recall  f1-score   support

           0       0.57      0.32      0.41       828
           1       0.59      0.81      0.68      1019

    accuracy                           0.59      1847
   macro avg       0.58      0.56      0.54      1847
weighted avg       0.58      0.59      0.56      1847



In [10]:
import pandas as pd

coef = best_lasso.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": diff_features,
    "coefficient": coef
}).sort_values("coefficient", ascending=False)

coef_df

,feature,coefficient
5,kddiff_diff,0.165786
4,kast_diff,0.134440
6,map_score_diff,0.117884
0,kills_diff,0.073819
3,adr_diff,0.011432
1,deaths_diff,0.000000
2,assists_diff,-0.101375


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import pandas as pd

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

grid_search_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [12]:
grid_search_rf.fit(X_train, y_train)

print("Best Random Forest parameters:", grid_search_rf.best_params_)
print("Best Random Forest CV accuracy:", grid_search_rf.best_score_)

# Test set prediction
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print("Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Best Random Forest parameters: {'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 100}
Best Random Forest CV accuracy: 0.5633801628949984
Random Forest Test Accuracy: 0.5728207904710341

Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.43      0.48       828
           1       0.60      0.69      0.64      1019

    accuracy                           0.57      1847
   macro avg       0.56      0.56      0.56      1847
weighted avg       0.57      0.57      0.57      1847


Confusion Matrix:
[[359 469]
 [320 699]]
